<a href="https://colab.research.google.com/github/AsimaZaheer/Stress-Detection/blob/main/Stress_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
"""
WESAD Stress Detection Pipeline — FIXED VERSION
=================================================
Original code mein 7 problems the, sab yahan fix kiye gaye hain.
Har fix ke paas comment hai jo batata hai KYA badla aur KYUN.

Run: Google Colab mein directly chal jayega (drive.mount already assumed).
"""

import os
import pickle
import numpy as np
from scipy import signal
from scipy.signal import find_peaks
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import accuracy_score, classification_report
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from google.colab import drive
drive.mount('/content/drive')

# ============================================================
# CONFIG — apni values yahan set karein
# ============================================================
DATASET_PATH = "/content/drive/MyDrive/WESAD/WESAD"
SUBJECTS = [f"S{i}" for i in [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17]]  # S1, S12 WESAD mein standard exclude hote hain

TARGET_SAMPLING_RATE = 4     # Hz -- EDA ki native wrist rate, BVP/labels isi pe align honge
LABEL_NATIVE_RATE = 700      # Hz -- WESAD ke chest label stream ki rate
WINDOW_SIZE = 60             # seconds
STEP_SIZE = 10               # seconds (83% overlap -> isliye LOSO zaroori hai, warna leakage)
BATCH_SIZE = 32
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# ============================================================
# HELPER: resampling
# ============================================================
def resample_signal(sig, orig_rate, target_rate):
    n_samples = int(len(sig) * target_rate / orig_rate)
    return signal.resample(sig, n_samples)


# ============================================================
# FIX 6: Label alignment — linspace stretching ki jagah
# actual sampling-rate-based timestamp mapping
# ------------------------------------------------------------
# PEHLE: np.linspace(0, len(labels_raw)-1, len(eda_resampled)) sirf do
# endpoints ke beech blind stretch karta tha — agar chest (700Hz) aur wrist
# recording ki length thodi bhi mismatch ho (jo WESAD mein hoti hai), to
# poora label stream silently compress/expand ho jata hai aur transition
# boundaries (baseline->stress) shift ho jaate hain.
# AB: har target sample ka ASLI time (seconds) nikal kar, usi time pe
# 700Hz label stream ka index nikalte hain. Isse mapping physically
# meaningful rehta hai aur transitions par error kam hota hai.
# ============================================================
def align_labels_by_time(labels_raw, label_rate, n_target_samples, target_rate):
    target_times = np.arange(n_target_samples) / target_rate
    label_indices = np.round(target_times * label_rate).astype(int)
    label_indices = np.clip(label_indices, 0, len(labels_raw) - 1)
    return labels_raw[label_indices]


# ============================================================
# FIX 4: HR / HRV features from BVP via peak detection
# ------------------------------------------------------------
# Pehle sirf raw EDA + BVP samples model ko diye ja rahe the. BVP se
# heartbeat peaks detect karke RR-intervals nikalte hain, jinse HR
# (heart rate) aur HRV (RMSSD, SDNN) banate hain — yeh teesri, physiologically
# distinct modality hai jo stress/amusement discrimination mein kaafi
# madad karti hai (autonomic nervous system signal).
# ============================================================
def extract_hrv_features(bvp_window, fs):
    peaks, _ = find_peaks(bvp_window, distance=max(1, int(fs * 0.4)), prominence=0.1)
    if len(peaks) < 3:
        return np.array([np.nan, np.nan, np.nan])

    rr_intervals = np.diff(peaks) / fs          # seconds
    hr = 60.0 / rr_intervals
    mean_hr = np.mean(hr)
    sdnn = np.std(rr_intervals) * 1000.0         # ms
    rmssd = np.sqrt(np.mean(np.diff(rr_intervals) ** 2)) * 1000.0  # ms
    return np.array([mean_hr, sdnn, rmssd])


# ============================================================
# FIX 1: Windowing function — syntax error fixed +
# HRV features integrated
# ------------------------------------------------------------
# `if majority_label in:` incomplete tha -> `if majority_label in [1, 2, 3]:`
# taake sirf valid WESAD labels (1=Baseline, 2=Stress, 3=Amusement) ke
# windows rakhe jayein, baaki (transient/meditation/etc, label 0,4,5,6,7)
# discard ho jayein.
# ============================================================
def create_windows_with_features(eda, bvp, labels, fs, window_size_s, step_size_s):
    window_samples = int(window_size_s * fs)
    step_samples = int(step_size_s * fs)

    X_seq, X_hrv, y = [], [], []

    for i in range(0, len(eda) - window_samples, step_samples):
        eda_win = eda[i:i + window_samples]
        bvp_win = bvp[i:i + window_samples]
        label_win = labels[i:i + window_samples]

        unique_labels, counts = np.unique(label_win, return_counts=True)
        majority_label = unique_labels[np.argmax(counts)]

        if majority_label in [1, 2, 3]:          # <-- FIX 1
            hrv_feats = extract_hrv_features(bvp_win, fs)
            if np.isnan(hrv_feats).any():
                continue  # itne kam peaks wala noisy window drop

            seq_window = np.column_stack((eda_win, bvp_win))
            X_seq.append(seq_window)
            X_hrv.append(hrv_feats)
            y.append(majority_label - 1)         # 1/2/3 -> 0/1/2 for CrossEntropyLoss

    return np.array(X_seq), np.array(X_hrv), np.array(y)


# ============================================================
# Simple model: 1D-CNN over (EDA, BVP) sequence + HRV features concat
# ============================================================
class StressClassifier(nn.Module):
    def __init__(self, seq_channels=2, hrv_dim=3, n_classes=3):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(seq_channels, 16, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(16, 32, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.classifier = nn.Sequential(
            nn.Linear(32 + hrv_dim, 32),
            nn.ReLU(),
            nn.Linear(32, n_classes),
        )

    def forward(self, x_seq, x_hrv):
        # x_seq: (batch, window_len, channels) -> conv1d wants (batch, channels, length)
        x = x_seq.permute(0, 2, 1)
        x = self.conv(x).squeeze(-1)          # (batch, 32)
        x = torch.cat([x, x_hrv], dim=1)      # (batch, 32 + hrv_dim)
        return self.classifier(x)


# ============================================================
# FIX 7: Poora pipeline ek reusable class mein wrap kiya —
# loading, alignment, windowing, LOSO loop, scaling sab clearly
# separated methods hain taake debug/samajhna aasan ho.
# ============================================================
class WESADPipeline:
    def __init__(self, dataset_path, subjects,
                 fs=TARGET_SAMPLING_RATE, label_rate=LABEL_NATIVE_RATE,
                 window_size=WINDOW_SIZE, step_size=STEP_SIZE):
        self.dataset_path = dataset_path
        self.subjects = subjects
        self.fs = fs
        self.label_rate = label_rate
        self.window_size = window_size
        self.step_size = step_size

    def load_subject_raw(self, sub):
        """Ek subject ki pkl file load karke EDA/BVP ko target_fs pe resample
        aur labels ko time-based align karta hai (FIX 6)."""
        file_path = os.path.join(self.dataset_path, sub, f"{sub}.pkl")
        if not os.path.exists(file_path):
            print(f"⚠️ Error: {file_path} nahi mili, skip kar rahe hain.")
            return None

        with open(file_path, 'rb') as f:
            data = pickle.load(f, encoding='latin1')

        eda_raw = data['signal']['wrist']['EDA'].flatten()
        bvp_raw = data['signal']['wrist']['BVP'].flatten()
        labels_raw = data['label'].flatten()

        eda_resampled = resample_signal(eda_raw, 4, self.fs)     # EDA native = 4Hz
        bvp_resampled = resample_signal(bvp_raw, 64, self.fs)    # BVP native = 64Hz

        n_target = min(len(eda_resampled), len(bvp_resampled))
        eda_resampled = eda_resampled[:n_target]
        bvp_resampled = bvp_resampled[:n_target]

        labels_aligned = align_labels_by_time(
            labels_raw, self.label_rate, n_target, self.fs
        )

        return eda_resampled, bvp_resampled, labels_aligned

    def build_all_subject_windows(self):
        """Har subject ke liye windows banata hai aur subject-id group label
        ke sath return karta hai (LOSO ke liye zaroori)."""
        X_seq_list, X_hrv_list, y_list, groups_list = [], [], [], []

        for sub in self.subjects:
            loaded = self.load_subject_raw(sub)
            if loaded is None:
                continue
            eda, bvp, labels = loaded

            X_seq, X_hrv, y = create_windows_with_features(
                eda, bvp, labels, self.fs, self.window_size, self.step_size
            )
            if len(y) == 0:
                continue

            X_seq_list.append(X_seq)
            X_hrv_list.append(X_hrv)
            y_list.append(y)
            groups_list.append(np.full(len(y), sub))

            print(f"  {sub}: {len(y)} windows")

        X_seq_all = np.concatenate(X_seq_list, axis=0)
        X_hrv_all = np.concatenate(X_hrv_list, axis=0)
        y_all = np.concatenate(y_list, axis=0)
        groups_all = np.concatenate(groups_list, axis=0)

        return X_seq_all, X_hrv_all, y_all, groups_all

    def run_loso(self, epochs=15, lr=1e-3, device=DEVICE):
        """
        FIX 2: Random train_test_split ki jagah Leave-One-Subject-Out CV.
        Sliding windows 83% overlap karte hain, isliye ek hi subject ke
        windows train aur test dono mein aa sakte the (near-duplicate data)
        -> yeh fake-high accuracy deta tha (leakage). Ab har fold mein
        EK poora subject sirf test ke liye alag rakha jata hai, kabhi bhi
        train mein uska koi window nahi aata.
        """
        print("⏳ Sab subjects load/window ho rahe hain...")
        X_seq_all, X_hrv_all, y_all, groups_all = self.build_all_subject_windows()

        # ---- FIX 5: class imbalance check ----
        unique, counts = np.unique(y_all, return_counts=True)
        print("\n📊 Overall class distribution (0=Baseline, 1=Stress, 2=Amusement):")
        print(dict(zip(unique, counts)))

        logo = LeaveOneGroupOut()
        fold_reports = []

        for fold_idx, (train_idx, test_idx) in enumerate(
                logo.split(X_seq_all, y_all, groups_all)):

            test_subject = groups_all[test_idx][0]
            print(f"\n===== FOLD {fold_idx + 1}/{len(self.subjects)} | Test subject: {test_subject} =====")

            X_seq_train, X_seq_test = X_seq_all[train_idx], X_seq_all[test_idx]
            X_hrv_train, X_hrv_test = X_hrv_all[train_idx], X_hrv_all[test_idx]
            y_train, y_test = y_all[train_idx], y_all[test_idx]

            # ------------------------------------------------------------
            # FIX 3: Scaler leakage fix.
            # PEHLE: StandardScaler poore dataset (train+test dono) pe
            # fit_transform ho raha tha -> test subject ki mean/std training
            # mein "leak" ho jaati thi.
            # AB: scaler sirf is fold ke training subjects ke data pe
            # fit hota hai; test subject (jo model ne kabhi nahi dekha)
            # pe sirf .transform() lagta hai.
            # ------------------------------------------------------------
            n_train, wlen, n_ch = X_seq_train.shape
            n_test = X_seq_test.shape[0]

            seq_scaler = StandardScaler()
            X_seq_train_s = seq_scaler.fit_transform(
                X_seq_train.reshape(-1, n_ch)).reshape(n_train, wlen, n_ch)
            X_seq_test_s = seq_scaler.transform(
                X_seq_test.reshape(-1, n_ch)).reshape(n_test, wlen, n_ch)

            hrv_scaler = StandardScaler()
            X_hrv_train_s = hrv_scaler.fit_transform(X_hrv_train)
            X_hrv_test_s = hrv_scaler.transform(X_hrv_test)

            # ---- FIX 5 (contd.): class weights for imbalance ----
            class_counts_train = np.bincount(y_train, minlength=3).astype(np.float32)
            class_weights = class_counts_train.sum() / (len(class_counts_train) * np.clip(class_counts_train, 1, None))
            class_weights_t = torch.tensor(class_weights, dtype=torch.float32, device=device)
            print("  Train class counts:", dict(zip(['Baseline', 'Stress', 'Amusement'], class_counts_train.astype(int))))
            print("  Class weights used in loss:", np.round(class_weights, 3))

            # ---- tensors + loaders ----
            train_ds = TensorDataset(
                torch.tensor(X_seq_train_s, dtype=torch.float32),
                torch.tensor(X_hrv_train_s, dtype=torch.float32),
                torch.tensor(y_train, dtype=torch.long),
            )
            test_ds = TensorDataset(
                torch.tensor(X_seq_test_s, dtype=torch.float32),
                torch.tensor(X_hrv_test_s, dtype=torch.float32),
                torch.tensor(y_test, dtype=torch.long),
            )
            train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
            test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

            # ---- model / train / eval for this fold ----
            model = StressClassifier().to(device)
            optimizer = torch.optim.Adam(model.parameters(), lr=lr)
            criterion = nn.CrossEntropyLoss(weight=class_weights_t)

            model.train()
            for epoch in range(epochs):
                total_loss = 0.0
                for xb_seq, xb_hrv, yb in train_loader:
                    xb_seq, xb_hrv, yb = xb_seq.to(device), xb_hrv.to(device), yb.to(device)
                    optimizer.zero_grad()
                    out = model(xb_seq, xb_hrv)
                    loss = criterion(out, yb)
                    loss.backward()
                    optimizer.step()
                    total_loss += loss.item()
                if (epoch + 1) % 5 == 0 or epoch == epochs - 1:
                    print(f"    epoch {epoch+1}/{epochs} - loss: {total_loss/len(train_loader):.4f}")

            model.eval()
            all_preds, all_true = [], []
            with torch.no_grad():
                for xb_seq, xb_hrv, yb in test_loader:
                    xb_seq, xb_hrv = xb_seq.to(device), xb_hrv.to(device)
                    out = model(xb_seq, xb_hrv)
                    preds = out.argmax(dim=1).cpu().numpy()
                    all_preds.extend(preds)
                    all_true.extend(yb.numpy())

            acc = accuracy_score(all_true, all_preds)
            print(f"  ✅ Test subject {test_subject} accuracy: {acc:.3f}")
            print(classification_report(
                all_true, all_preds,
                labels=[0, 1, 2],
                target_names=['Baseline', 'Stress', 'Amusement'],
                zero_division=0
            ))

            fold_reports.append({
                "test_subject": test_subject,
                "accuracy": acc,
                "n_train": n_train,
                "n_test": n_test,
            })

        return fold_reports


# ============================================================
# RUN
# ============================================================
if __name__ == "__main__":
    pipeline = WESADPipeline(DATASET_PATH, SUBJECTS)
    results = pipeline.run_loso(epochs=15, lr=1e-3)

    accs = [r["accuracy"] for r in results]
    print("\n================ LOSO SUMMARY ================")
    for r in results:
        print(f"  {r['test_subject']}: acc={r['accuracy']:.3f}  (train={r['n_train']}, test={r['n_test']})")
    print(f"\nMean LOSO accuracy: {np.mean(accs):.3f}  (± {np.std(accs):.3f})")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
⏳ Sab subjects load/window ho rahe hain...
  S2: 212 windows
  S3: 215 windows
  S4: 216 windows
  S5: 221 windows
⚠️ Error: /content/drive/MyDrive/WESAD/WESAD/S6/S6.pkl nahi mili, skip kar rahe hain.
⚠️ Error: /content/drive/MyDrive/WESAD/WESAD/S7/S7.pkl nahi mili, skip kar rahe hain.
⚠️ Error: /content/drive/MyDrive/WESAD/WESAD/S8/S8.pkl nahi mili, skip kar rahe hain.
⚠️ Error: /content/drive/MyDrive/WESAD/WESAD/S9/S9.pkl nahi mili, skip kar rahe hain.
  S10: 227 windows
  S11: 223 windows
  S13: 222 windows
  S14: 222 windows
  S15: 223 windows
  S16: 222 windows
  S17: 227 windows

📊 Overall class distribution (0=Baseline, 1=Stress, 2=Amusement):
{np.int32(0): np.int64(1290), np.int32(1): np.int64(734), np.int32(2): np.int64(406)}

===== FOLD 1/15 | Test subject: S10 =====
  Train class counts: {'Baseline': np.int64(1172), 'Stress': np.int64(662), 'Amusem